# 💳 Billing Components

> Presentation-only billing UI components for post-login transactional pages

In [ ]:
#| default_exp billing

## 🎯 Overview

| Category | Component | Purpose |
|----------|-----------|--------|
| 🛒 Checkout | `CheckoutCard` | Post-login checkout with monthly/yearly toggle and POST form |
| ⏳ Trial | `TrialBanner` | Dashboard banner showing trial status (info/warning tone) |
| 📊 Status | `BillingStatusCard` | Subscription status card with configurable messaging |

---

## 🏗️ Architecture

```
┌─────────────────────────────────────────────────────────┐
│                 Post-Login Dashboard                     │
├─────────────────────────────────────────────────────────┤
│  TrialBanner (info or warning tone)                     │
│  ├─ Days remaining + optional CTA                       │
├─────────────────────────────────────────────────────────┤
│  BillingStatusCard                                      │
│  ├─ Plan label + status chip + period end + manage CTA  │
├─────────────────────────────────────────────────────────┤
│  CheckoutCard                                           │
│  ├─ Monthly/yearly toggle + savings chip                │
│  ├─ POST form with hidden fields + CTA                  │
└─────────────────────────────────────────────────────────┘
```

---

## 📋 When to Use Which Component

| Scenario | Component | Module |
|----------|-----------|--------|
| Public marketing page with multiple pricing tiers | `PricingSection` | `web_pages` |
| Post-login checkout for a single plan | `CheckoutCard` | `billing` |
| Dashboard banner showing trial countdown | `TrialBanner` | `billing` |
| Dashboard card showing subscription state | `BillingStatusCard` | `billing` |

`PricingSection` is for **pre-login marketing**. The billing components are for **post-login transactional UI**.
All billing components are presentation-only and backend-agnostic — no payment provider integration, no hardcoded product names.

In [ ]:
#| export
from uuid import uuid4
from fh_matui.components import *
from fh_matui.core import *
from fh_matui.foundations import *
from fasthtml.common import *

In [ ]:
#| code-fold: true
#| eval: false

import socket
import time
import subprocess
from fastcore.utils import partial
from fasthtml.jupyter import FastHTML, JupyUvi, HTMX

def kill_process_on_port(port):
    """Kill any process using the specified port on Windows"""
    try:
        result = subprocess.run(
            f'netstat -ano | findstr :{port}',
            shell=True, capture_output=True, text=True
        )
        if result.stdout:
            lines = result.stdout.strip().split('\n')
            for line in lines:
                if 'LISTENING' in line:
                    pid = line.strip().split()[-1]
                    subprocess.run(f'taskkill /PID {pid} /F', shell=True, capture_output=True)
                    print(f"Killed process {pid} on port {port}")
                    time.sleep(0.5)
                    return True
        return False
    except Exception as e:
        print(f"Could not kill process on port {port}: {e}")
        return False

def find_available_port(start_port=5000, max_attempts=10):
    for port in range(start_port, start_port + max_attempts):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(('', port))
                return port
            except OSError:
                continue
    raise RuntimeError(f"Could not find available port in range {start_port}-{start_port+max_attempts}")

if 'server' in globals():
    try:
        server.stop()
        time.sleep(0.5)
    except:
        pass

preferred_port = 7030
kill_process_on_port(preferred_port)
port = find_available_port(preferred_port)

app = FastHTML(hdrs=MatTheme.blue.headers(title="Billing Components", mode="dark"))
rt = app.route

try:
    server = JupyUvi(app, port=port)
    preview = partial(HTMX, app=app, port=port)
    print(f"Server running on port {port}")
except Exception as e:
    print(f"Failed to start server: {e}")
    raise

Server running on port 7030


---

## 🛒 CheckoutCard

| Component | Purpose |
|-----------|--------|
| `CheckoutCard` | Single-plan checkout card with monthly/yearly toggle and POST form |

**Features:** Instance-safe JS IDs (uuid), configurable copy, hidden form fields, savings chip, trial callout

In [ ]:
#| export
def _checkout_toggle_css(uid: str) -> str:
    """Scoped CSS for checkout toggle — all selectors namespaced by uid."""
    return f"""
@keyframes checkout-pulse-{uid} {{
    0%, 100% {{ transform: scale(1); }}
    50% {{ transform: scale(1.05); }}
}}
.checkout-{uid}-savings {{ display: none; }}
.checkout-{uid}-savings.active {{
    display: inline-flex;
    animation: checkout-pulse-{uid} 0.3s ease-out;
}}
.checkout-{uid}-monthly {{ display: block; }}
.checkout-{uid}-yearly {{ display: none; }}
.checkout-{uid}-yearly.active {{ display: block; }}
.checkout-{uid}-monthly.active {{ display: block; }}
.checkout-{uid}-yearly:not(.active) {{ display: none; }}
.checkout-{uid}-monthly:not(.active) {{ display: none; }}
"""


def _checkout_toggle_js(uid: str) -> str:
    """Scoped JS for checkout toggle — function and selectors namespaced by uid."""
    return f"""
function toggleCheckout_{uid}(period) {{
    var monthly = document.querySelectorAll('.checkout-{uid}-monthly');
    var yearly = document.querySelectorAll('.checkout-{uid}-yearly');
    var chip = document.querySelector('.checkout-{uid}-savings');
    var btnM = document.querySelector('.checkout-{uid}-toggle-monthly');
    var btnY = document.querySelector('.checkout-{uid}-toggle-yearly');
    var hidden = document.getElementById('{uid}-billing-period');
    if (period === 'yearly') {{
        monthly.forEach(function(el) {{ el.classList.remove('active'); }});
        yearly.forEach(function(el) {{ el.classList.add('active'); }});
        if (chip) chip.classList.add('active');
        if (btnM) {{ btnM.classList.remove('fill'); btnM.classList.add('border'); }}
        if (btnY) {{ btnY.classList.add('fill'); btnY.classList.remove('border'); }}
    }} else {{
        yearly.forEach(function(el) {{ el.classList.remove('active'); }});
        monthly.forEach(function(el) {{ el.classList.add('active'); }});
        if (chip) chip.classList.remove('active');
        if (btnY) {{ btnY.classList.remove('fill'); btnY.classList.add('border'); }}
        if (btnM) {{ btnM.classList.add('fill'); btnM.classList.remove('border'); }}
    }}
    if (hidden) hidden.value = period;
}}
"""


def _calc_savings_pct(monthly_price: float, yearly_price: float) -> int:
    """Calculate savings percentage for yearly vs monthly billing."""
    if monthly_price <= 0:
        return 0
    annual_monthly = monthly_price * 12
    return max(0, round(100 - (yearly_price / annual_monthly) * 100))


def CheckoutCard(
    monthly_price: float,                 # Monthly price (required)
    yearly_price: float,                  # Yearly price (required)
    title: str = "Choose Your Plan",      # Card title
    subtitle: str = "",                   # Card subtitle
    currency: str = "$",                  # Currency symbol
    trial_days: int = 0,                  # Trial period in days (0 = none)
    cta_text: str = "Subscribe",          # Submit button text
    form_action: str = "/checkout",       # POST form action URL
    plan_id: str = "",                    # Hidden field: plan identifier
    hidden_fields: dict = None,           # Additional hidden fields {name: value}
    default_period: str = "monthly",      # Default toggle: 'monthly' or 'yearly'
    fine_print: str = "",                 # Small text below CTA
    cls: str = "",                        # Additional CSS classes
    **kwargs,                             # Pass-through to outer Article
):
    """Single-plan checkout card with monthly/yearly toggle and POST form.

    Designed for post-login checkout flows. Submits via POST form with hidden fields
    (plan_id, billing_period, plus any custom fields). All JS/CSS IDs are namespaced
    per instance using a uuid, so multiple CheckoutCards can coexist on one page.

    Args:
        monthly_price: Monthly price as float (e.g., 9.99)
        yearly_price: Yearly price as float (e.g., 99.99)
        title: Card heading
        subtitle: Card subheading
        currency: Currency symbol prepended to prices
        trial_days: If > 0, shows a trial callout above the form
        cta_text: Text for the submit button
        form_action: URL the form POSTs to
        plan_id: Value for the hidden plan_id field
        hidden_fields: Dict of additional hidden form fields {name: value}
        default_period: Which period is selected on load ('monthly' or 'yearly')
        fine_print: Small disclaimer text below the button
        cls: Additional CSS classes for the card

    Example:
        CheckoutCard(
            monthly_price=9.99,
            yearly_price=99.99,
            plan_id="pro",
            trial_days=14,
            fine_print="Cancel anytime. No questions asked.",
        )
    """
    uid = uuid4().hex[:8]
    savings_pct = _calc_savings_pct(monthly_price, yearly_price)

    # --- Toggle bar ---
    is_yearly = default_period == "yearly"
    m_cls = "fill" if not is_yearly else "border"
    y_cls = "fill" if is_yearly else "border"

    toggle_bar = Div(
        Nav(
            Button(
                "Monthly",
                cls=f"checkout-{uid}-toggle-monthly left-round {m_cls} small",
                onclick=f"toggleCheckout_{uid}('monthly')",
            ),
            Button(
                "Yearly",
                cls=f"checkout-{uid}-toggle-yearly right-round {y_cls} small",
                onclick=f"toggleCheckout_{uid}('yearly')",
            ),
        ),
        cls="row center-align middle-align small-space",
    )

    # --- Savings chip ---
    savings_active = "active" if is_yearly else ""
    savings_chip = Span(
        f"Save {savings_pct}%",
        cls=f"chip small green white-text checkout-{uid}-savings {savings_active}".strip(),
    ) if savings_pct > 0 else ""

    # --- Price display ---
    monthly_fmt = f"{currency}{monthly_price:.2f}"
    yearly_fmt = f"{currency}{yearly_price:.2f}"
    m_active = "active" if not is_yearly else ""
    y_active = "active" if is_yearly else ""

    price_display = Div(
        Div(
            H2(monthly_fmt, cls="center-align no-margin"),
            P("per month", cls="center-align secondary-text"),
            cls=f"checkout-{uid}-monthly {m_active}".strip(),
        ),
        Div(
            H2(yearly_fmt, cls="center-align no-margin"),
            P("per year", cls="center-align secondary-text"),
            cls=f"checkout-{uid}-yearly {y_active}".strip(),
        ),
    )

    # --- Trial callout ---
    trial_el = P(
        Icon("star", cls="primary-text"),
        f" {trial_days}-day free trial included",
        cls="center-align",
    ) if trial_days > 0 else ""

    # --- POST form with hidden fields ---
    form_children = [
        Input(type="hidden", name="plan_id", value=plan_id),
        Input(type="hidden", name="billing_period", value=default_period,
              id=f"{uid}-billing-period"),
    ]
    if hidden_fields:
        for fname, fval in hidden_fields.items():
            form_children.append(
                Input(type="hidden", name=str(fname), value=str(fval))
            )
    form_children.append(
        Button(cta_text, type="submit", cls="responsive primary")
    )

    form = ft_hx('form')(
        *form_children,
        method="POST",
        action=form_action,
    )

    # --- Fine print ---
    fine_print_el = Small(fine_print, cls="secondary-text center-align") if fine_print else ""

    # --- Assemble card ---
    card_children = [
        Style(_checkout_toggle_css(uid)),
        Script(_checkout_toggle_js(uid)),
    ]
    if title:
        card_children.append(H4(title, cls="center-align"))
    if subtitle:
        card_children.append(P(subtitle, cls="center-align secondary-text"))
    card_children.extend([
        toggle_bar,
        Div(savings_chip, cls="center-align small-margin") if savings_chip else "",
        price_display,
        trial_el,
        form,
        fine_print_el,
    ])

    return Article(
        *[c for c in card_children if c],
        cls=f"padding round surface-container {cls}".strip(),
        **kwargs,
    )

In [ ]:
#| code-fold: true
#| eval: false

# Example: CheckoutCard with trial period and custom hidden fields
def ex_checkout_card():
    return Div(
        CheckoutCard(
            monthly_price=9.99,
            yearly_price=99.99,
            plan_id="pro",
            trial_days=14,
            hidden_fields={"coupon": "WELCOME10"},
            fine_print="Cancel anytime. No questions asked.",
        ),
        cls="responsive padding",
    )

preview(ex_checkout_card())

In [ ]:
#| code-fold: true
#| eval: false

# Example: CheckoutCard defaulting to yearly billing
def ex_checkout_yearly():
    return Div(
        CheckoutCard(
            monthly_price=29.99,
            yearly_price=249.99,
            title="Enterprise Plan",
            subtitle="For growing teams",
            default_period="yearly",
            cta_text="Start Free Trial",
            form_action="/billing/subscribe",
            plan_id="enterprise",
        ),
        cls="responsive padding",
    )

preview(ex_checkout_yearly())

---

## ⏳ TrialBanner

| Component | Purpose |
|-----------|--------|
| `TrialBanner` | Thin, dismissible top-of-page bar showing trial countdown |

**Features:** Auto-derives info/warning tone, sits above the navbar, close button, configurable copy, optional CTA

**Positioning:** Place the banner **before** the navbar in the DOM. In a `TopLayout` route, 
return it as the first tuple element: `(TrialBanner(...), *TopLayout(content, nav_bar=...))`

In [ ]:
#| export
def TrialBanner(
    days_remaining: int,                    # Days left in trial
    end_date_text: str = "",                # e.g., 'Trial ends Mar 15'
    status: str = "trialing",               # Status key (for custom styling hooks)
    cta_text: str = "",                     # Optional CTA link text
    cta_href: str = "",                     # CTA link URL
    message: str = "",                      # Override default message
    dismissible: bool = True,               # Show close button
    banner_id: str = "",                    # Custom element ID (auto-generated if empty)
    cls: str = "",                          # Additional CSS classes
    **kwargs,                               # Pass-through to outer Div
):
    """Thin, dismissible trial banner — sits above the app layout.

    Renders a compact full-width bar with icon, message, optional CTA, and
    close button. Place it **before** the Layout in the DOM so it appears
    above everything. When dismissed, it removes itself from the page.

    Automatically switches to warning tone (error-container) when 3 or fewer
    days remain, otherwise uses info tone (primary-container).

    Positioning with Layout::

        @rt("/dashboard")
        def get(req):
            content = dashboard_content()
            if 'HX-Request' in req.headers: return content
            return (
                TrialBanner(days_remaining=5, cta_text="Upgrade", cta_href="/checkout"),
                Layout(content, sidebar_links=my_sidebar_links()),
            )

    Args:
        days_remaining: Number of days left in the trial
        end_date_text: Optional text (shown after a separator)
        status: Status string (available as a data attribute hook)
        cta_text: If non-empty, renders an inline CTA link
        cta_href: URL for the CTA link
        message: Custom message (overrides the default)
        dismissible: Whether to show the close button (default True)
        banner_id: Element ID for the banner (auto-generated if empty)
        cls: Additional CSS classes

    Example:
        TrialBanner(days_remaining=14, cta_text='Upgrade Now', cta_href='/checkout')
        TrialBanner(days_remaining=2, end_date_text='Trial ends Feb 23')
    """
    uid = uuid4().hex[:8]
    el_id = banner_id or f"trial-banner-{uid}"

    # Auto-derive tone
    is_warning = days_remaining <= 3
    tone_cls = "error-container" if is_warning else "primary-container"
    icon_name = "warning" if is_warning else "info"

    # Default message
    if not message:
        message = f"{days_remaining} day{'s' if days_remaining != 1 else ''} remaining in your trial"

    # Build inline content: icon + message + optional end date + optional CTA
    parts = [
        Icon(icon_name, cls="small"),
        Span(message, cls="small-text bold"),
    ]
    if end_date_text:
        parts.extend([
            Span("·", cls="small-text"),
            Span(end_date_text, cls="small-text"),
        ])
    if cta_text:
        parts.append(
            A(cta_text, href=cta_href, cls="small-text bold underline")
        )

    # Spacer pushes close button to the right
    parts.append(Div(cls="max"))

    # Close button
    if dismissible:
        parts.append(
            Button(
                Icon("close", cls="small"),
                cls="transparent circle small",
                onclick=f"document.getElementById('{el_id}').remove()",
                aria_label="Dismiss",
            )
        )

    return Div(
        Div(*parts, cls="row middle-align center-align small-space"),
        id=el_id,
        cls=f"{tone_cls} small-padding {cls}".strip(),
        style="z-index:10;",
        data_status=status,
        **kwargs,
    )

In [ ]:
#| code-fold: true
#| eval: false

# Example: TrialBanner in info tone (14 days remaining)
preview(TrialBanner(
    days_remaining=14,
    end_date_text="Trial ends March 15, 2026",
    cta_text="Upgrade Now",
    cta_href="/checkout",
))

In [ ]:
#| code-fold: true
#| eval: false

# Example: TrialBanner in warning tone (2 days remaining, non-dismissible)
preview(TrialBanner(
    days_remaining=2,
    end_date_text="Trial ends February 23, 2026",
    cta_text="Upgrade Now",
    cta_href="/checkout",
    dismissible=False,
))

---

## 📊 BillingStatusCard

| Component | Purpose |
|-----------|--------|
| `BillingStatusCard` | Subscription status card with status-specific messaging |

**Features:** Status chip with semantic colors, configurable messages, optional manage CTA

In [ ]:
#| export
_DEFAULT_BILLING_STATUS_MESSAGES = {
    "active": "Your subscription is active.",
    "trialing": "You're currently on a free trial.",
    "past_due": "Payment failed. Please update your payment method.",
    "canceled": "Your subscription has been canceled.",
    "checkout_pending": "Complete checkout to activate your plan.",
}

_BILLING_STATUS_COLORS = {
    "active": "green white-text",
    "trialing": "primary",
    "past_due": "error",
    "canceled": "grey white-text",
    "checkout_pending": "amber",
}


def BillingStatusCard(
    status: str,                            # 'active', 'trialing', 'past_due', 'canceled', 'checkout_pending'
    plan_label: str = "",                   # e.g., 'Pro Plan'
    current_period_end: str = "",           # e.g., 'March 15, 2026'
    manage_url: str = "",                   # URL for billing management portal
    manage_text: str = "Manage Billing",    # CTA button text
    messages: dict = None,                  # Override status -> message mapping
    cls: str = "",                          # Additional CSS classes
    **kwargs,                               # Pass-through to Card
):
    """Subscription status card with status-specific messaging.

    Displays the current subscription status with a semantic color chip,
    plan label, period end date, and an optional "Manage Billing" CTA.
    All copy is configurable via the messages dict override.

    Supported statuses: active, trialing, past_due, canceled, checkout_pending.
    Unknown statuses render with a neutral chip and the status string as message.

    Args:
        status: Subscription status string
        plan_label: Name of the current plan
        current_period_end: Human-readable period end date
        manage_url: If non-empty, renders a manage billing button
        manage_text: Text for the manage billing button
        messages: Dict overriding default status messages
        cls: Additional CSS classes

    Example:
        BillingStatusCard(status='active', plan_label='Pro Plan',
                          current_period_end='March 15, 2026',
                          manage_url='/billing/portal')
    """
    msgs = {**_DEFAULT_BILLING_STATUS_MESSAGES, **(messages or {})}
    status_msg = msgs.get(status, status.replace('_', ' ').title())
    chip_color = _BILLING_STATUS_COLORS.get(status, "secondary")
    chip_label = status.replace('_', ' ').title()

    # --- Header: plan label + status chip ---
    header_parts = []
    if plan_label:
        header_parts.append(H5(plan_label, cls="no-margin"))
    header_parts.append(Div(cls="max"))  # spacer
    header_parts.append(
        Span(chip_label, cls=f"chip small {chip_color}")
    )
    header = Div(*header_parts, cls="row middle-align")

    # --- Body: status message + period end ---
    body_parts = [P(status_msg)]
    if current_period_end:
        body_parts.append(
            P(f"Current period ends: {current_period_end}", cls="secondary-text small-text")
        )

    # --- Footer: manage CTA ---
    footer = None
    if manage_url:
        footer = A(manage_text, href=manage_url, cls="button primary small")

    return Card(
        *body_parts,
        header=header,
        footer=footer,
        cls=f"round {cls}".strip(),
        **kwargs,
    )

In [ ]:
#| code-fold: true
#| eval: false

# Example: BillingStatusCard for each status
def ex_billing_status_all():
    statuses = [
        ("active", "Pro Plan", "March 15, 2026"),
        ("trialing", "Pro Plan", "March 1, 2026"),
        ("past_due", "Pro Plan", "February 15, 2026"),
        ("canceled", "Pro Plan", ""),
        ("checkout_pending", "", ""),
    ]
    cards = [
        BillingStatusCard(
            status=s,
            plan_label=label,
            current_period_end=end,
            manage_url="/billing/portal" if s in ("active", "past_due") else "",
        )
        for s, label, end in statuses
    ]
    return Div(*cards, cls="responsive padding column medium-space")

preview(ex_billing_status_all())

---

## 🌐 Live App Preview

Full billing dashboard inside a `Layout` shell (sidebar + NavBar) with the `TrialBanner`
sitting above the entire layout. After running the server cell and this cell, open the port URL:

- `/billing` — Dashboard with TrialBanner + BillingStatusCard + CheckoutCard
- `/billing/checkout` — Centered CheckoutCard with trial callout
- `/billing/status` — All 5 subscription status cards + warning TrialBanner

In [ ]:
#| code-fold: true
#| eval: false

# --- Navigation items ---
def billing_nav_items():
    """Top NavBar links — hx-boost on Layout handles HTMX automatically."""
    return [
        A("Dashboard", href="/billing"),
        A("Checkout", href="/billing/checkout"),
        A("Status", href="/billing/status"),
    ]

def billing_sidebar_items():
    """Sidebar links with icons — hx-boost on Layout handles HTMX automatically."""
    return [
        A(Icon("dashboard"), Span("Dashboard"), href="/billing", cls="nav-link"),
        A(Icon("shopping_cart"), Span("Checkout"), href="/billing/checkout", cls="nav-link"),
        A(Icon("receipt_long"), Span("Status"), href="/billing/status", cls="nav-link"),
    ]

# Highlights the active sidebar link after each HTMX navigation
active_nav_script = Script("""
function updateActiveNav() {
    document.querySelectorAll('.nav-link').forEach(link => {
        link.classList.toggle('active', link.getAttribute('href') === window.location.pathname);
    });
}
if (document.readyState === 'loading') {
    document.addEventListener('DOMContentLoaded', updateActiveNav);
} else {
    updateActiveNav();
}
document.body.addEventListener('htmx:afterSettle', updateActiveNav);
""")


# --- App shell: TrialBanner sits above the full Layout ---
def get_billing_layout(content, days_remaining=5):
    banner = TrialBanner(
        days_remaining=days_remaining,
        end_date_text="Trial ends February 26, 2026",
        cta_text="Upgrade Now",
        cta_href="/billing/checkout",
    )
    return Div(
        banner,
        Layout(
            content,
            sidebar_links=billing_sidebar_items(),
            nav_bar=NavBar(*billing_nav_items(), brand=H3("BillingDemo"), sticky=True),
        ),
        active_nav_script,
    )


# --- Page content builders ---
def billing_dashboard_content():
    """Main billing dashboard: status card + checkout side-by-side."""
    return Div(
        Grid(
            GridCell(
                BillingStatusCard(
                    status="trialing",
                    plan_label="Pro Plan",
                    current_period_end="February 26, 2026",
                    manage_url="/billing/status",
                ),
                span="s12 m6",
            ),
            GridCell(
                CheckoutCard(
                    monthly_price=19.99,
                    yearly_price=199.99,
                    plan_id="pro",
                    trial_days=5,
                    cta_text="Subscribe Now",
                    form_action="/billing/checkout",
                ),
                span="s12 m6",
            ),
        ),
        cls="medium-space",
    )

def billing_checkout_content():
    """Dedicated checkout page with a single centered CheckoutCard."""
    return Grid(
        GridCell(span="s0 m2 l3"),
        GridCell(
            H3("Complete Your Purchase"),
            CheckoutCard(
                monthly_price=9.99,
                yearly_price=99.99,
                plan_id="pro",
                trial_days=14,
                hidden_fields={"coupon": "WELCOME10"},
                fine_print="Cancel anytime. No questions asked.",
                form_action="/billing/checkout",
            ),
            span="s12 m8 l6",
        ),
        GridCell(span="s0 m2 l3"),
    )

def billing_status_content():
    """All five subscription states for visual QA."""
    statuses = [
        ("active",           "Pro Plan", "March 15, 2026"),
        ("trialing",         "Pro Plan", "March 1, 2026"),
        ("past_due",         "Pro Plan", "February 15, 2026"),
        ("canceled",         "Pro Plan", ""),
        ("checkout_pending", "",         ""),
    ]
    cards = [
        BillingStatusCard(
            status=s, plan_label=label, current_period_end=end,
            manage_url="/billing" if s in ("active", "past_due") else "",
        )
        for s, label, end in statuses
    ]
    return Div(
        H3("All Subscription States"),
        Div(*cards, cls="medium-space"),
        cls="medium-space",
    )


# --- Routes ---
@rt("/billing")
def get(req):
    content = billing_dashboard_content()
    if "HX-Request" in req.headers: return content
    return get_billing_layout(content)

@rt("/billing/checkout")
def get(req):
    content = billing_checkout_content()
    if "HX-Request" in req.headers: return content
    return get_billing_layout(content)

@rt("/billing/status")
def get(req):
    content = billing_status_content()
    if "HX-Request" in req.headers: return content
    return get_billing_layout(content, days_remaining=2)  # warning tone on status page


# In-notebook preview
preview(get_billing_layout(billing_dashboard_content()))

---

## 🔍 Mobile & Theme Verification

All billing components use BeerCSS semantic classes (`primary-container`, `error-container`,
`surface-container`, etc.) so they automatically adapt to light/dark mode.

### Manual Checks

1. **Mobile (320–480px):** All cards stack full-width, toggle buttons remain tappable, form is usable
2. **Light mode:** Remove `dark` class from `<body>` — verify contrast and readability
3. **Dark mode:** Add `dark` class to `<body>` — verify all semantic colors adapt
4. **Form payload:** Open browser dev tools → Network tab → submit form → verify `plan_id`, `billing_period`, and custom hidden fields in POST body

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()